# Impedance ML Engineering — Session 18
## Turn a notebook model into a reproducible inference artifact
**Core: 60 minutes · ML engineering · Optional extension: 20–40 minutes**

## Why this matters
Can a colleague reproduce predictions from raw current-visit inputs without your notebook state?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Define an input/output schema and package preprocessing with weights.
- Separate training from prediction and preserve feature order.
- Verify round-trip predictions and meaningful failure checks.
- Record provenance, experiments, and model limitations.

## R-to-Python / prerequisite bridge
An R fitted object is not the whole workflow: save preprocessing, architecture, weights, schema, units, and versions; PyTorch `.state_dict()` contains parameters, not your preprocessing.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
A deployable inference artifact includes the fitted input transformations, estimator/weights, input contract, output meaning, and provenance. Saving weights alone is incomplete. Inference should not fit transformations or read outcomes. Reproducibility means another process can produce the same aligned predictions under the recorded environment.

A round-trip check tests the observable behavior of saving/reloading, not whether a serialized file merely exists. Column order, masks, categorical handling, and model mode are part of correctness. Only load trusted artifacts you produced.

Useful APIs: `joblib.dump`/`joblib.load`, `json.dump`, `hashlib.sha256`, `Path`, `subprocess.run`. For PyTorch save a `state_dict` with architecture/preprocessing config, load it into that architecture, then set eval and use no_grad for inference.

## Task 1 — Choose one reproducible candidate (5 min)
Use a fixed summary ridge as the core model, training only on 54 training patients. Create `src/impedance_ml/` and `artifacts/18/`. Declare inputs (11 named current columns) and output (next diagonal mean in kΩ). Predict shapes for batch sizes 1 and 5. A neural artifact is an optional extension.

## Task 2 — Separate fit from predict (8 min)
Move your pipeline construction/training into a function and define a separate `predict(frame,artifact)` function. Importing the module must not train a model or read arbitrary working-directory files. Keep IDs only for alignment in returned predictions, not inside the estimator. Use explicit paths/config arguments.

## Task 3 — Specify and validate the input schema (7 min)
Check required column names, numeric/categorical roles, and one row per current visit. Define whether missing numeric values are accepted by imputers. Reject future target columns if callers attempt to provide them as predictors. Reject missing required fields clearly; preserve documented input order before invoking the preprocessor.

## Task 4 — Save the fitted pipeline and metadata (8 min)
Persist the complete sklearn pipeline using joblib and save metadata JSON: target, schema, units, training data hashes, patient allocation, seed, versions, hyperparameters, and timestamp. Use only your own trusted local artifact. No network service is needed. Avoid claiming the validation score is a production guarantee.

## Task 5 — Verify a reload round trip (8 min)
In a fresh process reload the pipeline and schema, predict the same five validation inputs, and compare with original predictions using np.allclose. Check both row order and values. Demonstrate that the saved transformer medians/categories are used, not recomputed during inference.

## Task 6 — Write meaningful invariants (8 min)
Write checks for required-column failure, shuffled input columns handled according to schema, batch size 1 versus batch size 5 consistency, and finite output on accepted missing age. Add a check that training and validation/test patients are disjoint. Test behavior, not a mirror of implementation lines.

## Task 7 — Create a local batch interface (8 min)
Create an entry point that accepts an input CSV path and artifact directory, returns visit IDs and predictions to an explicit output path, and raises a useful error for malformed inputs. Run it on five validation rows. Do not silently overwrite source data or make inference inspect future outcomes.

## Task 8 — Write a model card and experiment record (0–2 min)
Document intended use, synthetic-data provenance, target population, metrics, excluded clinical claims, input QC, and retraining/monitoring questions. Record one command that reproduces training and one that reproduces prediction. This artifact is a learning deliverable, not an authorized deployed clinical system.

## Retrieval practice
Name everything missing if someone saves only CNN weights and later calls `.load_state_dict()` on a guessed architecture.

## Professional reasoning
1. What could go wrong if column order changes silently at inference?
2. Why must model.eval be restored for neural inference?
3. Which data/version changes should invalidate a comparison?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Package your small MLP or CNN with architecture config, state_dict, frozen preprocessing statistics, eval/no_grad inference, and a fresh-process round-trip check.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)